# 🇫🇷 Exploration des données PPA France

Ce notebook explore et visualise toutes les données générées par `download_france_data.py`.

**Prérequis :** Avoir exécuté `download_france_data.py` au préalable.

```bash
pip install plotly nbformat openpyxl
```

In [3]:
import pandas as pd
import numpy as np
import sqlite3
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Chemin vers votre dossier database (à adapter si nécessaire)
DB_PATH = "../French_PPA/database"  # relatif depuis votre notebook
# Ou chemin absolu :
# DB_PATH = r"C:/Users/cypri/OneDrive/Documents/1. Pro/Formations/Jupyter Notebooks/pyPPA/French_PPA/database"

print("✅ Imports OK")

✅ Imports OK


---
## 1. 📊 Grid France — CO2, Mix ENR, CAPEX
**Fichier :** `grid_france.csv`  
**Statut :** 🟡 Synthétique (projections ADEME/RTE)

In [5]:
import os
print(os.getcwd())

c:\Users\cypri\OneDrive\Documents\1. Pro\Formations\Jupyter Notebooks\pyPPA\French_PPA


In [4]:
grid_df = pd.read_csv(f"{DB_PATH}/grid_france.csv", index_col=0)
print("📋 Aperçu grid_france.csv :")
print(f"   Années : {grid_df.index.min()} → {grid_df.index.max()}")
print(f"   Colonnes : {list(grid_df.columns)}")
display(grid_df.head(10))

FileNotFoundError: [Errno 2] No such file or directory: '../French_PPA/database/grid_france.csv'

In [ ]:
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        "Intensité CO2 du réseau (gCO2/kWh)",
        "Part ENR dans le mix (%)",
        "CAPEX Solaire (k€/MW)",
        "CAPEX Éolien Offshore (k€/MW)"
    ]
)

# CO2
fig.add_trace(go.Scatter(
    x=grid_df.index, y=grid_df['co2'],
    name='CO2 (gCO2/kWh)', line=dict(color='#e74c3c', width=2),
    fill='tozeroy', fillcolor='rgba(231,76,60,0.1)'
), row=1, col=1)

# ENR share
fig.add_trace(go.Scatter(
    x=grid_df.index, y=grid_df['ren_share'] * 100,
    name='ENR (%)', line=dict(color='#27ae60', width=2),
    fill='tozeroy', fillcolor='rgba(39,174,96,0.1)'
), row=1, col=2)

# CAPEX Solaire
fig.add_trace(go.Scatter(
    x=grid_df.index, y=grid_df['solar_capex'] / 1000,
    name='CAPEX PV (k€/MW)', line=dict(color='#f39c12', width=2)
), row=2, col=1)

# CAPEX Éolien
fig.add_trace(go.Scatter(
    x=grid_df.index, y=grid_df['wind_capex'] / 1000,
    name='CAPEX Éolien (k€/MW)', line=dict(color='#2980b9', width=2)
), row=2, col=2)

fig.update_layout(
    title="📈 Projections Grid France 2023–2050",
    height=600, showlegend=False,
    template='plotly_white'
)
fig.show()

print(f"\n💡 Note : CO2 France (55 gCO2/kWh) vs Corée (~450 gCO2/kWh)")
print(f"   La France est déjà très décarbonée grâce au nucléaire.")

---
## 2. ☀️ Profils Solaires PVGIS
**Fichier :** `solar_patterns.db`  
**Statut :** 🟢 Données RÉELLES (PVGIS JRC, Dunkerque 2020)

In [ ]:
conn = sqlite3.connect(f"{DB_PATH}/solar_patterns.db")

# Lister les tables disponibles
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)
print(f"📋 Tables dans solar_patterns.db : {tables['name'].tolist()}")

solar_df = pd.read_sql("SELECT * FROM solar_patterns", conn)
conn.close()

solar_df['datetime'] = pd.to_datetime(solar_df['datetime'])
solar_df = solar_df.set_index('datetime')

print(f"\n📊 Statistiques du profil solaire (Dunkerque 2020) :")
print(f"   Nombre d'heures : {len(solar_df):,}")
print(f"   CF moyen annuel : {solar_df['q99'].mean():.3f} ({solar_df['q99'].mean()*100:.1f}%)")
print(f"   CF max (heure de pointe) : {solar_df['q99'].max():.3f}")
print(f"   Heures de production > 10% : {(solar_df['q99'] > 0.1).sum():,}")
display(solar_df.describe())

In [ ]:
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        "Profil horaire — Semaine type été (Juin)",
        "Profil horaire — Semaine type hiver (Décembre)",
        "Distribution mensuelle du CF solaire",
        "Heatmap solaire (heure × mois)"
    ]
)

# Semaine type été
ete = solar_df[solar_df.index.month == 6].iloc[:168]  # 1 semaine
fig.add_trace(go.Scatter(
    x=list(range(len(ete))), y=ete['q99'],
    name='Été (Juin)', line=dict(color='#f39c12'), fill='tozeroy'
), row=1, col=1)

# Semaine type hiver
hiver = solar_df[solar_df.index.month == 12].iloc[:168]
fig.add_trace(go.Scatter(
    x=list(range(len(hiver))), y=hiver['q99'],
    name='Hiver (Décembre)', line=dict(color='#2980b9'), fill='tozeroy'
), row=1, col=2)

# Distribution mensuelle
monthly_cf = solar_df.groupby(solar_df.index.month)['q99'].mean() * 100
month_names = ['Jan', 'Fév', 'Mar', 'Avr', 'Mai', 'Jun',
               'Jul', 'Aoû', 'Sep', 'Oct', 'Nov', 'Déc']
fig.add_trace(go.Bar(
    x=month_names, y=monthly_cf.values,
    name='CF mensuel (%)',
    marker_color=['#2980b9']*3 + ['#f39c12']*6 + ['#2980b9']*3
), row=2, col=1)

# Heatmap heure × mois
pivot = solar_df.copy()
pivot['hour'] = pivot.index.hour
pivot['month'] = pivot.index.month
heatmap_data = pivot.groupby(['hour', 'month'])['q99'].mean().unstack()

fig.add_trace(go.Heatmap(
    z=heatmap_data.values,
    x=month_names,
    y=[f"{h}h" for h in range(24)],
    colorscale='YlOrRd',
    showscale=True,
    name='Heatmap CF'
), row=2, col=2)

fig.update_layout(
    title="☀️ Profil Solaire PVGIS — Dunkerque 2020 (Données RÉELLES)",
    height=700, showlegend=False,
    template='plotly_white'
)
fig.show()

---
## 3. 💨 Profils Éoliens Open-Meteo/ERA5
**Fichier :** `wind_patterns.db`  
**Statut :** 🟢 Données RÉELLES (ERA5 réanalyse ECMWF)

In [ ]:
conn = sqlite3.connect(f"{DB_PATH}/wind_patterns.db")
tables_wind = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)
print(f"📋 Tables dans wind_patterns.db : {tables_wind['name'].tolist()}")

wind_df = pd.read_sql("SELECT * FROM wind_patterns", conn)
conn.close()

wind_df['index'] = pd.to_datetime(wind_df['index'])
wind_df = wind_df.set_index('index')

print(f"\n📊 Statistiques par région :")
print(wind_df.describe().round(3))

In [ ]:
regions = [c for c in wind_df.columns]
colors = ['#2980b9', '#27ae60', '#e74c3c', '#f39c12', '#9b59b6']

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        "CF moyen mensuel par région",
        "Distribution des CF (boîte à moustaches)",
        "Profil éolien — Semaine type (Janvier)",
        "Courbe de durée éolienne annuelle"
    ]
)

# CF mensuel par région
for i, (region, color) in enumerate(zip(regions, colors)):
    monthly = wind_df[region].groupby(wind_df.index.month).mean() * 100
    fig.add_trace(go.Scatter(
        x=month_names, y=monthly.values,
        name=region, line=dict(color=color, width=2),
        mode='lines+markers'
    ), row=1, col=1)

# Boxplot distribution
for i, (region, color) in enumerate(zip(regions, colors)):
    fig.add_trace(go.Box(
        y=wind_df[region] * 100,
        name=region,
        marker_color=color,
        showlegend=False
    ), row=1, col=2)

# Semaine type janvier
janvier = wind_df[wind_df.index.month == 1].iloc[:168]
for i, (region, color) in enumerate(zip(regions, colors)):
    fig.add_trace(go.Scatter(
        x=list(range(len(janvier))), y=janvier[region],
        name=region, line=dict(color=color),
        showlegend=False
    ), row=2, col=1)

# Courbe de durée
for i, (region, color) in enumerate(zip(regions, colors)):
    sorted_cf = np.sort(wind_df[region].values)[::-1] * 100
    hours = np.arange(1, len(sorted_cf) + 1)
    fig.add_trace(go.Scatter(
        x=hours, y=sorted_cf,
        name=region, line=dict(color=color),
        showlegend=False
    ), row=2, col=2)

fig.update_layout(
    title="💨 Profils Éoliens Offshore — France 2020 (Données RÉELLES ERA5)",
    height=700, template='plotly_white',
    legend=dict(orientation='h', y=-0.15)
)
fig.update_yaxes(title_text="CF (%)", row=1, col=1)
fig.update_yaxes(title_text="CF (%)", row=1, col=2)
fig.update_xaxes(title_text="Heures (rang)", row=2, col=2)
fig.show()

# Tableau récapitulatif
print("\n📊 Capacité Factor annuel moyen par région :")
summary = pd.DataFrame({
    'CF moyen (%)': (wind_df.mean() * 100).round(1),
    'CF max (%)': (wind_df.max() * 100).round(1),
    'Heures > 50%': (wind_df > 0.5).sum(),
    'Heures > 80%': (wind_df > 0.8).sum(),
})
display(summary)

---
## 4. 🏭 Profil de Charge Industriel
**Fichier :** `load_patterns.db`  
**Statut :** 🟡 Synthétique (profil semi-continu générique)

In [ ]:
conn = sqlite3.connect(f"{DB_PATH}/load_patterns.db")
load_df = pd.read_sql("SELECT * FROM load_patterns", conn)
conn.close()

load_df['datetime'] = pd.to_datetime(load_df['datetime'])
load_df = load_df.set_index('datetime')

print(f"📊 Profil de charge industriel :")
print(f"   CF moyen : {load_df['value'].mean():.3f} ({load_df['value'].mean()*100:.1f}%)")
print(f"   CF min : {load_df['value'].min():.3f}")
print(f"   CF max : {load_df['value'].max():.3f}")

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=["Profil type 2 semaines (Janv-Fév)", "Distribution des charges"]
)

# 2 semaines
two_weeks = load_df.iloc[:336]
fig.add_trace(go.Scatter(
    x=two_weeks.index, y=two_weeks['value'] * 100,
    fill='tozeroy', line=dict(color='#8e44ad'),
    name='Charge (%)'
), row=1, col=1)

# Histogramme
fig.add_trace(go.Histogram(
    x=load_df['value'] * 100,
    nbinsx=30, marker_color='#8e44ad',
    name='Distribution'
), row=1, col=2)

fig.update_layout(
    title="🏭 Profil de Charge Industriel (Synthétique — À remplacer par données client)",
    height=400, template='plotly_white', showlegend=False
)
fig.show()

print("\n⚠️  Ce profil est SYNTHÉTIQUE.")
print("   Demandez à votre client sa courbe de charge annuelle réelle")
print("   (disponible via son gestionnaire énergie ou son contrat ENEDIS).")

---
## 5. ⚡ Tarifs Réseau TURPE (KEPCO_france.xlsx)
**Fichier :** `KEPCO_france.xlsx`  
**Statut :** 🟡 Synthétique (valeurs CRE approximées)

In [ ]:
# Lire toutes les sheets
xl = pd.ExcelFile(f"{DB_PATH}/KEPCO_france.xlsx")
print(f"📋 Sheets disponibles : {xl.sheet_names}")

for sheet in xl.sheet_names:
    df = pd.read_excel(f"{DB_PATH}/KEPCO_france.xlsx", sheet_name=sheet)
    print(f"\n--- Sheet '{sheet}' ---")
    display(df)

In [ ]:
# Reconstituer le profil tarifaire horaire TURPE pour 2030
import sys
sys.path.insert(0, '..')  # Ajouter le dossier parent si FranceGridUtils est là

# Construction manuelle du profil pour visualisation
year = 2030
date_range = pd.date_range(start=f"{year}-01-01", end=f"{year}-12-31 23:00", freq="h")

months = date_range.month
hours = date_range.hour
weekdays = date_range.weekday

is_winter = months.isin([11, 12, 1, 2, 3])
is_daytime = (hours >= 6) & (hours < 22)
is_weekday = weekdays < 5

# TURPE énergie
turpe_energy = np.zeros(len(date_range))
turpe_energy[is_winter & is_daytime & is_weekday] = 5.2   # HPH
turpe_energy[is_winter & ~(is_daytime & is_weekday)] = 1.8  # HCH
turpe_energy[~is_winter & is_daytime & is_weekday] = 1.0  # HPE
turpe_energy[~is_winter & ~(is_daytime & is_weekday)] = 0.5  # HCE

# EPEX synthétique (80 €/MWh moyen)
base = 80.0
seasonal = np.where(months.isin([12,1,2]), 1.30, np.where(months.isin([6,7,8]), 0.90, 1.0))
np.random.seed(42)
epex = base * seasonal * np.random.lognormal(0, 0.1, len(date_range))

total_rate = epex + turpe_energy + 0.5 + 0.3  # + TICFE + CTA

rate_df = pd.DataFrame({
    'datetime': date_range,
    'EPEX Spot': epex,
    'TURPE Énergie': turpe_energy,
    'Taxes (TICFE+CTA)': 0.8,
    'Total (€/MWh)': total_rate
}).set_index('datetime')

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        "Profil tarifaire — Semaine type hiver",
        "Profil tarifaire — Semaine type été",
        "Distribution mensuelle du tarif total",
        "Composition du tarif par composante"
    ]
)

# Hiver
hiver_rate = rate_df[rate_df.index.month == 1].iloc[:168]
fig.add_trace(go.Scatter(x=list(range(168)), y=hiver_rate['Total (€/MWh)'],
    name='Tarif total', line=dict(color='#e74c3c')), row=1, col=1)
fig.add_trace(go.Scatter(x=list(range(168)), y=hiver_rate['EPEX Spot'],
    name='EPEX Spot', line=dict(color='#e74c3c', dash='dot')), row=1, col=1)

# Été
ete_rate = rate_df[rate_df.index.month == 7].iloc[:168]
fig.add_trace(go.Scatter(x=list(range(168)), y=ete_rate['Total (€/MWh)'],
    name='Tarif total', line=dict(color='#f39c12'), showlegend=False), row=1, col=2)

# Monthly box
monthly_rate = rate_df.groupby(rate_df.index.month)['Total (€/MWh)'].mean()
fig.add_trace(go.Bar(
    x=month_names, y=monthly_rate.values,
    marker_color=['#2980b9']*3 + ['#27ae60']*6 + ['#2980b9']*3,
    showlegend=False
), row=2, col=1)

# Composition
components = ['EPEX Spot', 'TURPE Énergie', 'Taxes (TICFE+CTA)']
values = [rate_df['EPEX Spot'].mean(), rate_df['TURPE Énergie'].mean(), 0.8]
fig.add_trace(go.Pie(
    labels=components, values=values,
    hole=0.4
), row=2, col=2)

fig.update_layout(
    title="⚡ Structure du Tarif Électricité HTB3 — France 2030",
    height=650, template='plotly_white'
)
fig.show()

print(f"\n📊 Résumé tarifaire 2030 (€/MWh) :")
print(f"   Tarif total moyen : {total_rate.mean():.1f} €/MWh")
print(f"   EPEX moyen        : {epex.mean():.1f} €/MWh")
print(f"   TURPE énergie moy : {turpe_energy.mean():.1f} €/MWh")
print(f"   Tarif min/max     : {total_rate.min():.1f} / {total_rate.max():.1f} €/MWh")

---
## 6. 🌬️ Parcs Éoliens Offshore (wind_grid_france.xlsx)
**Fichier :** `wind_grid_france.xlsx`  
**Statut :** 🟡 Synthétique (6 parcs encodés manuellement)

In [ ]:
wind_grid = pd.read_excel(f"{DB_PATH}/wind_grid_france.xlsx")
print("📋 Parcs éoliens offshore français :")
display(wind_grid)

# Carte des parcs
fig = px.scatter_mapbox(
    wind_grid,
    lat='lat', lon='lon',
    size='capacity',
    color='LCOE',
    hover_name='nom_parc',
    hover_data={'capacity': True, 'LCOE': True, 'DT_m': True, 'admin_boundaries': True},
    color_continuous_scale='RdYlGn_r',
    size_max=30,
    zoom=5,
    center={'lat': 48.5, 'lon': -1.0},
    mapbox_style='open-street-map',
    title='🌬️ Parcs Éoliens Offshore France — LCOE (€/MWh) et Capacité'
)
fig.update_layout(height=500)
fig.show()

# Comparatif LCOE
fig2 = px.bar(
    wind_grid.sort_values('LCOE'),
    x='nom_parc', y='LCOE',
    color='admin_boundaries',
    text='capacity',
    title='Comparatif LCOE des parcs éoliens offshore (€/MWh)',
    labels={'nom_parc': 'Parc', 'LCOE': 'LCOE (€/MWh)', 'capacity': 'Capacité (MW)'}
)
fig2.update_traces(texttemplate='%{text} MW', textposition='outside')
fig2.update_layout(template='plotly_white', height=400)
fig2.show()

---
## 7. 🔄 Vue d'ensemble : Comparaison Solaire vs Éolien
Superposition des profils pour comprendre la complémentarité des sources

In [ ]:
# Recharger les données
conn_s = sqlite3.connect(f"{DB_PATH}/solar_patterns.db")
solar = pd.read_sql("SELECT * FROM solar_patterns", conn_s)
solar['datetime'] = pd.to_datetime(solar['datetime'])
solar = solar.set_index('datetime')
conn_s.close()

conn_w = sqlite3.connect(f"{DB_PATH}/wind_patterns.db")
wind = pd.read_sql("SELECT * FROM wind_patterns", conn_w)
wind['index'] = pd.to_datetime(wind['index'])
wind = wind.set_index('index')
conn_w.close()

# Aligner sur la même période
common_idx = solar.index.intersection(wind.index)
solar_aligned = solar.loc[common_idx, 'q99']

# Prendre la meilleure région éolienne (Occitanie)
best_wind = wind.loc[common_idx].max(axis=1)  # max toutes régions

# Semaine type hiver — complémentarité
week_winter = common_idx[common_idx.month == 1][:168]
week_summer = common_idx[common_idx.month == 7][:168]

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        "Complémentarité Solaire/Éolien — Hiver (1 semaine)",
        "Complémentarité Solaire/Éolien — Été (1 semaine)",
        "CF mensuel moyen — Comparaison sources",
        "Corrélation Solaire vs Éolien"
    ]
)

for week, row, season in [(week_winter, 1, 'Hiver'), (week_summer, 2, 'Été')]:
    col = 1 if row == 1 else 2
    # Solaire
    fig.add_trace(go.Scatter(
        x=list(range(len(week))), y=solar_aligned.loc[week].values * 100,
        name='Solaire', line=dict(color='#f39c12', width=2),
        fill='tozeroy', fillcolor='rgba(243,156,18,0.2)'
    ), row=1, col=col)
    # Éolien
    fig.add_trace(go.Scatter(
        x=list(range(len(week))), y=best_wind.loc[week].values * 100,
        name='Éolien (meilleure zone)', line=dict(color='#2980b9', width=2),
        fill='tozeroy', fillcolor='rgba(41,128,185,0.2)'
    ), row=1, col=col)

# Mensuel
monthly_solar = solar_aligned.groupby(solar_aligned.index.month).mean() * 100
monthly_wind = best_wind.groupby(best_wind.index.month).mean() * 100

fig.add_trace(go.Scatter(
    x=month_names, y=monthly_solar.values,
    name='Solaire', line=dict(color='#f39c12', width=3),
    mode='lines+markers', showlegend=False
), row=2, col=1)
fig.add_trace(go.Scatter(
    x=month_names, y=monthly_wind.values,
    name='Éolien', line=dict(color='#2980b9', width=3),
    mode='lines+markers', showlegend=False
), row=2, col=1)

# Scatter corrélation
sample_idx = np.random.choice(len(solar_aligned), 2000, replace=False)
fig.add_trace(go.Scatter(
    x=solar_aligned.iloc[sample_idx].values * 100,
    y=best_wind.iloc[sample_idx].values * 100,
    mode='markers',
    marker=dict(size=3, opacity=0.3, color='#27ae60'),
    name='Corrélation',
    showlegend=False
), row=2, col=2)

corr = np.corrcoef(solar_aligned.values, best_wind.values)[0, 1]

fig.update_layout(
    title=f"🔄 Complémentarité Solaire/Éolien (corrélation = {corr:.3f})",
    height=700, template='plotly_white'
)
fig.update_xaxes(title_text="Solaire CF (%)", row=2, col=2)
fig.update_yaxes(title_text="Éolien CF (%)", row=2, col=2)
fig.show()

print(f"\n💡 Corrélation solaire/éolien : {corr:.3f}")
print(f"   Une corrélation négative signifie bonne complémentarité.")
print(f"   Hiver = éolien fort + solaire faible → complémentaires ✅")
print(f"   Été   = solaire fort + éolien plus faible → complémentaires ✅")

---
## 8. 📋 Résumé de qualité des données
Tableau de synthèse pour savoir quoi améliorer en priorité

In [ ]:
summary_data = {
    'Fichier': [
        'solar_patterns.db',
        'wind_patterns.db',
        'grid_france.csv',
        'load_patterns.db',
        'KEPCO_france.xlsx',
        'wind_grid_france.xlsx'
    ],
    'Statut': ['🟢 Réel', '🟢 Réel', '🟡 Synthétique', '🟡 Synthétique', '🟡 Synthétique', '🟡 Synthétique'],
    'Source': [
        'PVGIS JRC (Dunkerque 2020)',
        'Open-Meteo ERA5 (5 régions 2020)',
        'Projections ADEME/RTE manuelles',
        'Profil industriel générique',
        'Valeurs CRE approximées',
        '6 parcs encodés manuellement'
    ],
    'Priorité amélioration': [
        '🔵 Basse (données de qualité)',
        '🔵 Basse (données de qualité)',
        '🟠 Haute (télécharger eco2mix ODRE)',
        '🔴 Critique (demander au client)',
        '🟠 Haute (vérifier sur cre.fr)',
        '🟠 Haute (données CRE appels offres)'
    ],
    'Où obtenir les données réelles': [
        'pvgis.ec.europa.eu (autre site → changer lat/lon)',
        'open-meteo.com (autre site ou période)',
        'odre.opendatasoft.com → eco2mix national',
        'Contrat énergie client ou gestionnaire',
        'cre.fr → TURPE 6 HTB délibération',
        'cre.fr → AO éolien en mer + thewindpower.net'
    ]
}

summary_df = pd.DataFrame(summary_data)

print("\n📋 SYNTHÈSE QUALITÉ DES DONNÉES\n")
pd.set_option('display.max_colwidth', 60)
display(summary_df.set_index('Fichier'))

print("\n🎯 Actions prioritaires :")
print("   1. [CRITIQUE] Récupérer la courbe de charge réelle du client industriel")
print("   2. [HAUTE]    Télécharger eco2mix ODRE pour CO2/ENR réels")
print("   3. [HAUTE]    Vérifier tarifs TURPE sur cre.fr (délibération en vigueur)")
print("   4. [BASSE]    Télécharger PVGIS pour le site exact du client (changer lat/lon)")
print("   5. [BASSE]    Télécharger vent ERA5 pour années supplémentaires")